<a href="https://colab.research.google.com/github/sandeepyelekar2/GenAIDeveloper_training/blob/Bronze_Badge_for_Gen_AIAssessment/Assignments_5_Policy_Claims_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Pin opentelemetry first to avoid a conflict with Colab's preinstalled google-adk package.
# Then install the LLM, vector DB, embeddings, UI, and data libraries.
!pip -q install opentelemetry-api==1.42.1 opentelemetry-sdk==1.42.1
!pip -q install openai chromadb sentence-transformers gradio pandas transformers accelerate sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
import os
import textwrap
import pandas as pd
import chromadb
from getpass import getpass
from openai import OpenAI, AuthenticationError
from sentence_transformers import SentenceTransformer

# Toggle: set USE_LOCAL_MODEL=1 (default) to run fully offline with a small open-source model.
# To use OpenAI in the same notebook later, set an environment variable in Colab before running:
#   %env USE_LOCAL_MODEL=0
USE_LOCAL_MODEL = os.getenv("USE_LOCAL_MODEL", "1") == "1"

# Shared model names
MODEL_NAME = "gpt-4o-mini"  # OpenAI remote model (used only when USE_LOCAL_MODEL is False)
LOCAL_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # local fallback model

client = None
local_pipe = None

if USE_LOCAL_MODEL:
    print("Using local offline model:", LOCAL_MODEL_NAME)
    # Build a local text-generation pipeline (no API key required)
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    try:
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_NAME)
        # device_map="auto" can be added if a GPU is available; Colab CPU works without it.
        local_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
    except Exception as e:
        print(f"Failed to initialize local model '{LOCAL_MODEL_NAME}': {e}")
        print("You can switch to OpenAI by setting %env USE_LOCAL_MODEL=0 and re-running the setup cell.")
else:
    # Online path: read & verify OpenAI API key, then create the client.
    def is_valid_key_format(key: str) -> bool:
        if not key:
            return False
        key = key.strip()
        if len(key) < 30:
            return False
        if key.lower() in ("password", "your-api-key", "sk-your-api-key", "none"):
            return False
        return key.startswith("sk-") or key.startswith("proj-")

    def get_api_key() -> str:
        key = os.getenv('OPENAI_API_KEY', '').strip()
        while True:
            if not is_valid_key_format(key):
                key = getpass('Enter your OpenAI API key (starts with sk- or proj-): ').strip()
                continue
            test_client = OpenAI(api_key=key)
            try:
                test_client.models.list()  # live verification
                print('API key verified successfully.')
                return key
            except AuthenticationError:
                print('Invalid API key. Get one from https://platform.openai.com/account/api-keys')
                key = ''
            except Exception as e:
                print(f'Could not verify key due to error: {e}')
                key = ''

    os.environ['OPENAI_API_KEY'] = get_api_key()
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# Embedding model for retrieval (works offline)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Using local offline model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
sample_policy = '''
# Sample Health Policy Knowledge Base

## Coverage
The policy covers medically necessary hospitalization expenses when the insured is admitted for at least 24 hours. Covered items include room rent, ICU charges, nursing charges, doctor consultation, surgery charges, anesthesia, medicines, diagnostic tests, and ambulance charges.

## Room rent limit
Room rent is covered up to 1% of the sum insured per day. ICU charges are covered up to 2% of the sum insured per day. If the insured chooses a higher room category, proportionate deductions may apply to associated medical expenses.

## Waiting periods
Initial waiting period is 30 days from policy start date, except accidental injury claims. Specific disease waiting period is 24 months for cataract, hernia, piles, sinusitis, joint replacement due to degenerative conditions, and gall bladder stone treatment. Pre-existing diseases have a waiting period of 36 months.

## Exclusions
Cosmetic treatment, experimental treatment, non-medical expenses, dental treatment unless caused by accident, fertility treatment, self-inflicted injury, and treatment outside policy geography are not covered. OPD consultation is not covered unless an OPD add-on is purchased.

## Sub-limits
Cataract treatment is covered up to 25% of sum insured or 40,000 per eye, whichever is lower, after the specific disease waiting period is completed. Ambulance charges are covered up to 2,000 per hospitalization. Daily cash benefit is covered only when the policy schedule includes this benefit.

## Claim submission steps
For cashless claims, the customer must approach a network hospital, show health card and ID proof, submit pre-authorization form, and wait for insurer approval. For reimbursement claims, the customer must pay hospital bills first and submit claim documents after discharge.

## Claim timelines
Cashless pre-authorization is usually processed within 4 hours after receiving complete documents. Reimbursement claims should be submitted within 15 days of discharge. Claim settlement is usually completed within 30 days after receiving all required documents and clarifications.

## Documents needed
Hospitalization claim documents include completed claim form, policy number, health card, government ID proof, discharge summary, final hospital bill, payment receipts, investigation reports, prescriptions, pharmacy bills, cancelled cheque, and KYC documents when required.
'''.strip()

policy_path = 'sample_policies/policy_knowledge_base.md'
if os.path.exists(policy_path):
    with open(policy_path, 'r', encoding='utf-8') as f:
        policy_text = f.read()
else:
    policy_text = sample_policy

print(policy_text[:700])

# Sample Health Policy Knowledge Base

## Coverage
The policy covers medically necessary hospitalization expenses when the insured is admitted for at least 24 hours. Covered items include room rent, ICU charges, nursing charges, doctor consultation, surgery charges, anesthesia, medicines, diagnostic tests, and ambulance charges.

## Room rent limit
Room rent is covered up to 1% of the sum insured per day. ICU charges are covered up to 2% of the sum insured per day. If the insured chooses a higher room category, proportionate deductions may apply to associated medical expenses.

## Waiting periods
Initial waiting period is 30 days from policy start date, except accidental injury claims. Speci


In [4]:
def chunk_text(text, chunk_size=700, overlap=120):
    # Character-based chunking is simple and reliable for a demo.
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return [chunk.strip() for chunk in chunks if chunk.strip()]

chunks = chunk_text(policy_text)
chunk_embeddings = embedding_model.encode(chunks).tolist()

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name='policy_claims_copilot')

# Reset the collection contents so notebook re-runs do not duplicate chunks.
existing = collection.get()
if existing.get('ids'):
    collection.delete(ids=existing['ids'])

collection.add(
    ids=[f'policy_chunk_{i}' for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings,
    metadatas=[{'source': 'sample_policy', 'chunk': i} for i in range(len(chunks))]
)

print(f'Indexed {len(chunks)} policy chunks')

Indexed 5 policy chunks


In [5]:
def retrieve_context(query, top_k=4):
    query_embedding = embedding_model.encode([query]).tolist()[0]
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    documents = results['documents'][0]
    metadatas = results['metadatas'][0]
    return [
        {'text': doc, 'source': meta.get('source'), 'chunk': meta.get('chunk')}
        for doc, meta in zip(documents, metadatas)
    ]

retrieve_context('What documents are needed for hospitalization claim?', top_k=2)

[{'text': 'he customer must pay hospital bills first and submit claim documents after discharge.\n\n## Claim timelines\nCashless pre-authorization is usually processed within 4 hours after receiving complete documents. Reimbursement claims should be submitted within 15 days of discharge. Claim settlement is usually completed within 30 days after receiving all required documents and clarifications.\n\n## Documents needed\nHospitalization claim documents include completed claim form, policy number, health card, government ID proof, discharge summary, final hospital bill, payment receipts, investigation reports, prescriptions, pharmacy bills, cancelled cheque, and KYC documents when required.',
  'source': 'sample_policy',
  'chunk': 3},
 {'text': 'estigation reports, prescriptions, pharmacy bills, cancelled cheque, and KYC documents when required.',
  'source': 'sample_policy',
  'chunk': 4}]

In [6]:
def answer_policy_question(question):
    # Retrieves relevant policy chunks and answers from them using either
    # the local offline model or OpenAI (based on USE_LOCAL_MODEL).
    retrieved = retrieve_context(question)
    context = '\n\n'.join([f"[chunk {r['chunk']}] {r['text']}" for r in retrieved])

    system_prompt = '''You are Policy & Claims Copilot.
Answer customer support questions using only the provided policy context.
If the answer is not present, say that the policy text does not provide enough information.
Use concise language and include citations like [chunk 2].'''

    if USE_LOCAL_MODEL and local_pipe is not None:
        # Single-string prompt for small local chat models
        prompt = (
            "System: " + system_prompt + "\n\n" +
            "Policy context:\n" + context + "\n\n" +
            "User: " + question + "\n\n" +
            "Assistant:"
        )
        out = local_pipe(prompt, max_new_tokens=512, do_sample=False, temperature=0.1)
        text = out[0]['generated_text']
        # Return the part after "Assistant:" to reduce echo.
        return text.split('Assistant:')[-1].strip()

    # Online path: OpenAI chat completions
    user_prompt = f'''Policy context:\n{context}\n\nQuestion: {question}\n\nAnswer:'''
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content

print(answer_policy_question('What documents are required for a hospitalization reimbursement claim?'))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress 

Sure, here's a sample health policy knowledge base:

Coverage: The policy covers medically necessary hospitalization expenses when the insured is admitted for at least 24 hours. Covered items include room rent, ICU charges, nursing charges, doctor consultation, surgery charges, anesthesia, medicines, diagnostic tests, and ambulance charges. Room rent is covered up to 1% of the sum insured per day. ICU charges are covered up to 2% of the sum insured per day. If the insured chooses a higher room category, proportionate deductions may apply to associated medical expenses. Waiting periods: Initial waiting period is 30 days from policy start date, except accidental injury claims. Speci

[chunk 2] ion is not covered unless an OPD add-on is purchased. Sub-limits: Cataract treatment is covered up to 25% of sum insured or 40,000 per eye, whichever is lower, after the specific disease waiting period is completed. Ambulance charges are covered up to 2,000 per hospitalization. Daily cash benefit i

In [9]:
def precheck_claim(scenario):
    # Retrieves relevant policy chunks and asks the LLM for a structured pre-check.
    retrieved = retrieve_context(scenario, top_k=5)
    context = '\n\n'.join([f"[chunk {r['chunk']}] {r['text']}" for r in retrieved])

    system_prompt = '''You are a claims pre-check assistant.
Use only the retrieved policy context.
Return a structured pre-check with these sections:
1. Preliminary decision: Likely eligible / Needs review / Likely not eligible
2. Coverage basis
3. Waiting-period concerns
4. Limits or sub-limits
5. Missing documents
6. Recommended next steps
Always include citations like [chunk 1]. This is not final claim approval.'''

    if USE_LOCAL_MODEL and local_pipe is not None:
        prompt = (
            "System: " + system_prompt + "\n\n" +
            "Retrieved policy context:\n" + context + "\n\n" +
            "Claim scenario:\n" + scenario + "\n\n" +
            "Pre-check result:\nAssistant:"
        )
        out = local_pipe(prompt, max_new_tokens=700, do_sample=False, temperature=0.1)
        text = out[0]['generated_text']
        return text.split('Assistant:')[-1].strip()

    user_prompt = f'''Retrieved policy context:\n{context}\n\nClaim scenario:\n{scenario}\n\nPre-check result:\n'''
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content

scenario = 'Customer has a 3-month-old health policy and needs cataract surgery claim of 80000. Documents: health card, ID proof, pre-authorization form.'
print(precheck_claim(scenario))

[transformers] Both `max_new_tokens` (=700) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Customer has a 3-month-old health policy and needs cataract surgery claim of 80000. Documents: health card, ID proof, pre-authorization form, policy number, hospitalization claim form,


In [10]:
claims_csv_path = 'sample_claims/sample_claim_scenarios.csv'
if os.path.exists(claims_csv_path):
    sample_claims = pd.read_csv(claims_csv_path)
else:
    sample_claims = pd.DataFrame([
        {'scenario_id': 'S001', 'policy_age_months': 18, 'treatment': 'hospitalization for dengue', 'claim_amount': 65000, 'admission_type': 'reimbursement', 'documents_available': 'claim form;health card;ID proof;discharge summary;final bill;payment receipts;reports;prescriptions;pharmacy bills;cancelled cheque'},
        {'scenario_id': 'S002', 'policy_age_months': 3, 'treatment': 'cataract surgery', 'claim_amount': 80000, 'admission_type': 'cashless', 'documents_available': 'health card;ID proof;pre-authorization form'},
        {'scenario_id': 'S003', 'policy_age_months': 40, 'treatment': 'dental implant', 'claim_amount': 45000, 'admission_type': 'reimbursement', 'documents_available': 'claim form;ID proof;final bill;payment receipts'}
    ])

def row_to_scenario(row):
    # Convert tabular claim data into a natural-language scenario for retrieval and LLM analysis.
    return (
        f"Policy age: {row.policy_age_months} months. "
        f"Treatment: {row.treatment}. Claim amount: {row.claim_amount}. "
        f"Admission type: {row.admission_type}. Documents available: {row.documents_available}."
    )

sample_claims['scenario_text'] = sample_claims.apply(row_to_scenario, axis=1)
sample_claims[['scenario_id', 'scenario_text']]

,scenario_id,scenario_text
0,S001,Policy age: 18 months. Treatment: hospitalizat...
1,S002,Policy age: 3 months. Treatment: cataract surg...
2,S003,Policy age: 40 months. Treatment: dental impla...


In [11]:
import gradio as gr

# Ensure the required functions are available (common cause after a runtime reset)
if 'answer_policy_question' not in globals() or 'precheck_claim' not in globals():
    raise RuntimeError(
        'Please run the previous cells first to define retrieval and model functions (Sections 4-7).\n'
        'Tip: use Runtime > Run all, then re-run this cell.'
    )

with gr.Blocks(title='Policy & Claims Copilot') as demo:
    gr.Markdown('# Policy & Claims Copilot')
    gr.Markdown('LLM + RAG assistant for policy Q&A and claims pre-check.')

    with gr.Tab('Policy Q&A'):
        question = gr.Textbox(label='Ask a policy question', placeholder='Example: What documents are needed for hospitalization claim?')
        answer = gr.Textbox(label='Answer with citations', lines=8)
        gr.Button('Ask').click(answer_policy_question, inputs=question, outputs=answer)

    with gr.Tab('Claim Pre-check'):
        scenario_input = gr.Textbox(label='Claim scenario', lines=5, placeholder='Example: 3-month-old policy, cataract surgery, claim amount 80000...')
        precheck_output = gr.Textbox(label='Pre-check result', lines=14)
        gr.Button('Run Pre-check').click(precheck_claim, inputs=scenario_input, outputs=precheck_output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7c04339fe94c2c6c2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
